In [1]:
import pandas as pd
import numpy as np

from typing import List, Optional
from dataclasses import dataclass
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random
import os

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score, average_precision_score
import optuna
import joblib
from typing import List, Tuple, Optional
import re

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df1 = pd.read_excel('data/Таблицы [06-02-2026 14-02-07] СПК 210 на СГП.xlsx', header=4, index_col=0)
df2 = pd.read_excel('data/Терморобот Сланцы.xlsx', header=4)
df3 = pd.read_excel('data/КВМ с ТШПМ Бирюсинск.xlsx', header=4, index_col=0)

In [3]:
df11 = df1.iloc[:, :4]
df12 = df1.iloc[:, 4:8]
df13 = df1.iloc[:, 8:]
# в числовой
df12['Камера 2 Влажность, %'] = pd.to_numeric(df12['Камера 2 Влажность, %'], errors='coerce')
df = df12.copy()
# убрать секунды
df.index = df.index.floor('min')
# удаление дубликатов
df = df[~df.index.duplicated(keep='first')]
# исправление некорректных значений(ошибки датчика)
df.loc[(df.iloc[:, 2] <= 0), 'Камера 2 Влажность, %'] = np.nan
df.loc[(df.iloc[:, 0] < -100)|(df.iloc[:, 0] > 100)|(df.iloc[:, 0] == 0), 'Камера 2 Температура °С'] = np.nan
# регулирование частоты
df = df.asfreq('1min')
# интерполяция
df = df.interpolate(method='linear')
df = df.iloc[:, [0, 2]]
df = df.reset_index()

In [67]:
class FeatureEngineer:
    """Генератор признаков для временных рядов"""
    
    def create_features(self, df_fe: pd.DataFrame, windows: List[int] = [5, 15, 30, 60]) -> pd.DataFrame:
        """Создание всех признаков"""
        
        # Копируем данные
        data = df_fe.copy()
        data = data.sort_values(TIME_COL)
        
        # 1. Базовые признаки
        #data['temp_to_min_threshold'] = data[TEMP_COL] - 2.0
        data['temp_to_max_threshold'] = -7.0 - data[TEMP_COL]
        data['hum_to_threshold'] = 80.0 - data[HUMID_COL]
        
        # 2. Производные первого порядка
        data['temp_diff'] = data[TEMP_COL].diff()
        data['hum_diff'] = data[HUMID_COL].diff()
        data['temp_change_rate'] = data['temp_diff'] / 1.0  # за 1 минуту
        data['hum_change_rate'] = data['hum_diff'] / 1.0
        
        # 3. Производные второго порядка
        data['temp_accel'] = data['temp_change_rate'].diff()
        data['hum_accel'] = data['hum_change_rate'].diff()
        
        # 4. Взаимодействие признаков
        data['dew_point_approx'] = data[TEMP_COL] - ((100 - data[HUMID_COL]) / 5)
        data['temp_hum_interaction'] = data[TEMP_COL] * data[HUMID_COL]
        
        # 5. Статистики в скользящих окнах
        for window in windows:
            # Температура
            data[f'temp_roll_mean_{window}'] = data[TEMP_COL].rolling(window=window, min_periods=1).mean()
            data[f'temp_roll_std_{window}'] = data[TEMP_COL].rolling(window=window, min_periods=1).std()
            data[f'temp_roll_min_{window}'] = data[TEMP_COL].rolling(window=window, min_periods=1).min()
            data[f'temp_roll_max_{window}'] = data[TEMP_COL].rolling(window=window, min_periods=1).max()
            
            # Тренд за окно (линейная регрессия)
            data[f'temp_trend_{window}'] = self._calculate_trend(data[TEMP_COL], window)
            
            # Скорость изменения
            data[f'temp_change_roll_{window}'] = data['temp_change_rate'].rolling(window=window, min_periods=1).mean()
            
            # Влажность
            data[f'hum_roll_mean_{window}'] = data[HUMID_COL].rolling(window=window, min_periods=1).mean()
            data[f'hum_roll_std_{window}'] = data[HUMID_COL].rolling(window=window, min_periods=1).std()
            
        # 6. Циклические временные признаки
        data['hour'] = data[TIME_COL].dt.hour
        data['minute'] = data[TIME_COL].dt.minute
        data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
        data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
        
        # 7. Логарифмические признаки (для стабилизации дисперсии)
        data['temp_log'] = np.log1p(data[TEMP_COL] - data[TEMP_COL].min() + 1)
        
        # Заполняем пропуски
        data = data.fillna(method='ffill').fillna(method='bfill')
        
        return data
    
    def _calculate_trend(self, series: pd.Series, window: int) -> pd.Series:
        """Расчет тренда линейной регрессией в скользящем окне"""
        def linear_trend(x):
            if len(x) < 2:
                return 0
            x = np.arange(len(x))
            slope, _ = np.polyfit(x, x, 1)
            return slope
        
        return series.rolling(window=window, min_periods=2).apply(linear_trend, raw=True)

In [ ]:
class LabelGenerator:
    """Создание меток для обучения с backtracing"""
    
    def __init__(self):
        self.forecast_horizon = FORECAST_HORIZON
        
    def create_labels(self, 
                     df_lg: pd.DataFrame):
        """Создание бинарных меток аномалий"""
        
        data = df_lg.copy()
        
        # 1. Определяем моменты реальных аварий
        data['is_failure'] = 0
        data.loc[(data[TEMP_COL] >= TEMP_THRESHOLD) |
                (data[HUMID_COL] >= HUMID_THRESHOLD), 'is_failure'] = 1
        
        # 2. Идентификация непрерывных блоков аварий
        #    Каждый переход 0→1 инициирует новый блок
        data['_block'] = (data['is_failure'].diff() == 1).cumsum()
        data.loc[data['is_failure'] == 0, '_block'] = 0

        # 3. Инициализация целевой переменной (по умолчанию 0)
        data['target'] = 0

        # 4. Для каждого блока ставим target = 1 на окне предупреждения
        for block_id in data[data['is_failure'] == 1]['_block'].unique():
            if block_id == 0:
                continue
            block_mask = data['_block'] == block_id
            start_idx = data[block_mask].index[0]          # временная метка начала аварии
            start_pos = data.index.get_loc(start_idx)      # позиционный индекс (int)
            left_pos = max(0, start_pos - self.forecast_horizon)

            # Устанавливаем 1 на [left_pos, start_pos) — НЕ включая момент start_pos!
            data.iloc[left_pos:start_pos, data.columns.get_loc('target')] = 1

        # 5. Удаляем служебную колонку
        data.drop(columns=['_block'], inplace=True)
        
        return data
    
    def create_regression_labels(self, df_lg: pd.DataFrame) -> pd.DataFrame:
        """Создание регрессионных меток: время до аварии в минутах"""
        
        data = df_lg.copy()
        data['time_to_failure'] = np.inf
        
        # Находим следующие аварии для каждой точки
        failure_indices = data[data['is_failure'] == 1].index
        
        for idx in failure_indices:
            # Для всех точек за forecast_horizon минут до аварии
            start_idx = max(0, idx - self.forecast_horizon)
            for i in range(start_idx, idx):
                time_to_fail = idx - i
                if time_to_fail < data.loc[data.index[i], 'time_to_failure']:
                    data.loc[data.index[i], 'time_to_failure'] = time_to_fail
                    
        # Преобразуем в нормальное распределение
        data['ttf_log'] = np.log1p(data['time_to_failure'])
        
        return data

In [29]:
train, test = df.iloc[:-2000], df.iloc[-2000:]

In [4]:
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()
split_idx = int(len(val_df) * 0.5)
val = val_df.iloc[split_idx:].copy()
test = val_df.iloc[split_idx:].copy()

In [71]:
class AnomalyPredictor:
    """
    Ансамбль LightGBM + LSTM для предсказания аварий за N минут до их начала.
    """

    def __init__(self, config: dict):
        self.config = config
        self.random_state = self.config['random_state']
        random.seed(self.random_state)
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.random_state)
            torch.cuda.manual_seed_all(self.random_state)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            #os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
            #torch.use_deterministic_algorithms(True)
        self.lgb_model = None
        self.lstm_model = None
        self.scaler = StandardScaler()
        self.feature_cols = None
        self.device = torch.device('cuda')

        # --- Извлекаем настройки ---
        self.temp_col = self.config['temp_col']
        self.humid_col = self.config['humid_col']
        self.time_col = self.config['time_col']
        self.temp_threshold = self.config['temp_threshold']
        self.humid_threshold = self.config['humid_threshold']
        self.forecast_horizon = self.config['forecast_horizon']

        # Параметры признаков
        self.rolling_windows = self.config['features'].get('rolling_windows', [5, 15, 30, 60])

        # Параметры LSTM
        self.seq_length = self.config['params_lstm']['seq_length']
        self.lstm_epochs = self.config['params_lstm']['lstm_epochs']
        self.lstm_patience = self.config['params_lstm']['lstm_patience']
        self.batch_size = self.config['params_lstm']['batch_size']

        # Параметры LightGBM
        self.lgb_params = self.config['params_lgbm']
        self.lgb_num_boost_round = self.config['lgb_num_boost_round']
        self.lgb_early_stopping = self.config['lgb_early_stopping']
        
    # 1. ВНУТРЕННИЙ КЛАСС – ГЕНЕРАТОР МЕТОК (бин. классификация)
    class _LabelGenerator:
        """Создаёт колонки is_failure (авария сейчас) и target (предупреждение за N мин)."""
        def __init__(self, outer):
            self.outer = outer

        def create_labels(self, df: pd.DataFrame) -> pd.DataFrame:
            data = df.copy()
            # 1. Метка «авария сейчас» (включая равенство порогу)
            data['is_failure'] = 0
            data.loc[(data[self.outer.temp_col] >= self.outer.temp_threshold) |
                     (data[self.outer.humid_col] >= self.outer.humid_threshold), 'is_failure'] = 1

            # 2. Идентификация непрерывных блоков аварий
            #data['_block'] = (data['is_failure'].diff() == 1).cumsum()
            #data.loc[data['is_failure'] == 0, '_block'] = 0

            # 3. Целевая переменная (по умолчанию 0)
            data['target'] = 0

            # 3. Получаем все позиции, где is_failure == 1
            failure_positions = data[data['is_failure'] == 1].index
            if len(failure_positions) == 0:
                return data

            # 4. Для каждой позиции аварии
            for pos in failure_positions:
                pos_idx = data.index.get_loc(pos)                # позиционный индекс (int)
                left = max(0, pos_idx - self.outer.forecast_horizon)
                # Ставим target = 1 на [left, pos_idx] включительно
                data.iloc[left:pos_idx + 1, data.columns.get_loc('target')] = 1

            # 5. Убираем единицы в самих моментах аварии
            data.loc[data['is_failure'] == 1, 'target'] = 0
            
            # 4. Для каждого блока – окно предупреждения [начало - horizon, начало)
            #for block_id in data[data['is_failure'] == 1]['_block'].unique():
            #    if block_id == 0:
            #        continue
            #    block_mask = data['_block'] == block_id
            #    start_idx = data[block_mask].index[0]
            #    start_pos = data.index.get_loc(start_idx)
            #    left_pos = max(0, start_pos - self.outer.forecast_horizon)
            #    data.iloc[left_pos:start_pos, data.columns.get_loc('target')] = 1

            #data.drop(columns=['_block'], inplace=True)
            return data

    # 2. ВНУТРЕННИЙ КЛАСС – ИНЖИНИРИНГ ПРИЗНАКОВ
    class _FeatureEngineer:
        """Генерация признаков (лаги, скользящие окна, тренды, временные признаки)."""
        def __init__(self, outer):
            self.outer = outer

        def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
            data = df.copy()
            data = data.sort_values(self.outer.time_col)

            # 1. Расстояние до порога
            data['temp_to_threshold'] = self.outer.temp_threshold - data[self.outer.temp_col]
            data['hum_to_threshold'] = self.outer.humid_threshold - data[self.outer.humid_col]

            # 2. Производные первого порядка
            data['temp_diff'] = data[self.outer.temp_col].diff()
            data['hum_diff'] = data[self.outer.humid_col].diff()
            data['temp_change_rate'] = data['temp_diff'] / 1.0
            data['hum_change_rate'] = data['hum_diff'] / 1.0

            # 3. Производные второго порядка
            data['temp_accel'] = data['temp_change_rate'].diff()
            data['hum_accel'] = data['hum_change_rate'].diff()

            # 4. Взаимодействия
            data['dew_point_approx'] = data[self.outer.temp_col] - ((100 - data[self.outer.humid_col]) / 5)
            data['temp_hum_interaction'] = data[self.outer.temp_col] * data[self.outer.humid_col]

            # 5. Скользящие окна
            for w in self.outer.rolling_windows:
                # Температура
                data[f'temp_roll_mean_{w}'] = data[self.outer.temp_col].rolling(w, min_periods=1).mean()
                data[f'temp_roll_std_{w}'] = data[self.outer.temp_col].rolling(w, min_periods=1).std()
                data[f'temp_roll_min_{w}'] = data[self.outer.temp_col].rolling(w, min_periods=1).min()
                data[f'temp_roll_max_{w}'] = data[self.outer.temp_col].rolling(w, min_periods=1).max()
                data[f'temp_trend_{w}'] = self._calc_trend(data[self.outer.temp_col], w)
                data[f'temp_change_roll_{w}'] = data['temp_change_rate'].rolling(w, min_periods=1).mean()
                # Влажность
                data[f'hum_roll_mean_{w}'] = data[self.outer.humid_col].rolling(w, min_periods=1).mean()
                data[f'hum_roll_std_{w}'] = data[self.outer.humid_col].rolling(w, min_periods=1).std()

            # 6. Временные признаки
            data['hour'] = data[self.outer.time_col].dt.hour
            data['minute'] = data[self.outer.time_col].dt.minute
            data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
            data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

            # 7. Логарифмический признак (стабилизация)
            data['temp_log'] = np.log1p(data[self.outer.temp_col] - data[self.outer.temp_col].min() + 1)

            # Заполняем пропуски
            data = data.ffill().bfill()
            return data

        @staticmethod
        def _calc_trend(series: pd.Series, window: int) -> pd.Series:
            """Линейный тренд в скользящем окне."""
            def _trend(y):
                if len(y) < 2:
                    return 0.0
                x = np.arange(len(y))
                slope, _ = np.polyfit(x, y, 1)
                return slope
            return series.rolling(window, min_periods=2).apply(_trend, raw=True)

    # 3. ПОДГОТОВКА ПРИЗНАКОВ И ЦЕЛЕВОЙ ПЕРЕМЕННОЙ
    def prepare_features(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
        """
        Полный конвейер: инжиниринг + разметка → очистка имён → удаление NaN.
        Возвращает (X, is_failure, target).
        """
        # Инжиниринг
        fe = self._FeatureEngineer(self)
        data = fe.create_features(df)

        # Разметка
        lg = self._LabelGenerator(self)
        data = lg.create_labels(data)

        data.rename(columns={self.temp_col:'temp' , self.humid_col: 'humid'}, inplace=True)
        
        # Отбор числовых признаков (исключаем служебные колонки)
        numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
        exclude = ['target', 'is_failure', 'time_to_failure', 'ttf_log']
        self.feature_cols = [c for c in numeric_cols if c not in exclude]
        
        # Удаляем NaN (появились из-за сдвигов/окон)
        #data.dropna(inplace=True)

        X = data[self.feature_cols]
        y_is_failure = data['is_failure']
        y_target = data['target']

        return X, y_is_failure, y_target

    # 4. LSTM – построение, обучение, создание последовательностей
    def build_lstm(self, input_shape: Tuple[int, int]):
        """Создаёт LSTM модель на PyTorch."""
        seq_len, n_feat = input_shape
        hidden_size1 = self.config['params_lstm']['lstm_hidden_size1']
        hidden_size2 = self.config['params_lstm']['lstm_hidden_size2']
        fc_size = self.config['params_lstm']['lstm_fc_size']
        dropout = self.config['params_lstm']['dropout']

        class LSTMPredictor(nn.Module):
            def __init__(self, sl, nf, hidden_size1, hidden_size2, fc_size, dropout):
                super().__init__()
                self.lstm1 = nn.LSTM(nf, hidden_size1, batch_first=True, dropout=dropout)
                self.lstm2 = nn.LSTM(hidden_size1, hidden_size2, batch_first=True, dropout=dropout)
                self.fc1 = nn.Linear(hidden_size2, fc_size)
                self.relu = nn.ReLU()
                self.dropout = nn.Dropout(dropout)
                self.fc2 = nn.Linear(fc_size, 1)
                self.sigmoid = nn.Sigmoid()
            def forward(self, x):
                out, _ = self.lstm1(x)
                out, _ = self.lstm2(out)
                out = out[:, -1, :]
                out = self.fc1(out)
                out = self.relu(out)
                out = self.dropout(out)
                out = self.fc2(out)
                #out = self.sigmoid(out)
                return out.squeeze(1)

        self.lstm_model = LSTMPredictor(
            seq_len, n_feat,
            hidden_size1=hidden_size1,
            hidden_size2=hidden_size2,
            fc_size=fc_size,
            dropout=dropout
        ).to(self.device)
        return self.lstm_model

    @staticmethod
    def create_sequences(data: np.ndarray, seq_length: int) -> np.ndarray:
        """(samples, features) → (samples - seq_len + 1, seq_len, features)."""
        seqs = []
        for i in range(len(data) - seq_length + 1):
            seqs.append(data[i:i+seq_length])
        return np.array(seqs)

    def train_lstm(self, X_train_seq, y_train_seq, X_val_seq, y_val_seq,
                   epochs=None, batch_size=None, patience=None, return_best_loss=True):
        """Обучение LSTM с early stopping и восстановлением лучших весов."""
        epochs = self.lstm_epochs
        batch_size = self.batch_size
        patience = self.lstm_patience
        pos_weight = torch.tensor([self.config['params_lstm']['lstm_pos_weight']]).to(self.device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # или BCELoss + вес вручную

        #criterion = nn.BCELoss()

        optimizer = optim.AdamW(self.lstm_model.parameters(), lr=0.001)

        train_ds = TensorDataset(torch.FloatTensor(X_train_seq).to(self.device),
                                 torch.FloatTensor(y_train_seq).to(self.device))
        val_ds = TensorDataset(torch.FloatTensor(X_val_seq).to(self.device),
                               torch.FloatTensor(y_val_seq).to(self.device))
        
        g = torch.Generator()
        g.manual_seed(self.random_state)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

        best_val_loss = float('inf')
        best_state = None
        patience_counter = 0

        for epoch in range(epochs):
            # Train
            self.lstm_model.train()
            train_loss = 0.0
            for Xb, yb in train_loader:
                optimizer.zero_grad()
                outputs = self.lstm_model(Xb)
                loss = criterion(outputs, yb)
                #weights = torch.where(yb == 1, pos_weight, 1.0)
                #loss = (loss_per_element * weights).mean()
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * Xb.size(0)
            train_loss /= len(train_loader.dataset)

            # Validation
            self.lstm_model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for Xb, yb in val_loader:
                    outputs = self.lstm_model(Xb)
                    loss = criterion(outputs, yb)
                    val_loss += loss.item() * Xb.size(0)
            val_loss /= len(val_loader.dataset)

            # Save best
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = self.lstm_model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"LSTM early stopping at epoch {epoch+1}, best val_loss: {best_val_loss:.4f}")
                    break

            if (epoch+1) % 5 == 0:
                print(f"LSTM Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, "
                      f"Val Loss: {val_loss:.4f} (best: {best_val_loss:.4f})")

        if best_state is not None:
            self.lstm_model.load_state_dict(best_state)
        if return_best_loss:
            return best_val_loss

    # 5. LIGHTGBM – обучение
    def train_lgbm(self, X_train, y_train, X_val, y_val):
        """Обучение LightGBM с балансировкой классов."""
        # Веса классов
        bc = np.bincount(y_train.astype(int))
        class_weights = len(y_train) / (2 * bc)
        weight_map = {i: class_weights[i] for i in range(len(class_weights))}
        sample_weights = y_train.map(weight_map)

        train_data = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

        self.lgb_model = lgb.train(
            self.lgb_params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=self.lgb_num_boost_round,
            callbacks=[lgb.early_stopping(self.lgb_early_stopping)]
        )
        return self.lgb_model

    # 6. ОБУЧЕНИЕ АНСАМБЛЯ (с кросс-валидацией и выбором лучших моделей)
    def train_ensemble(self, df_train: pd.DataFrame):
        """Основной метод обучения: временная кросс-валидация, сохранение лучших весов в памяти."""
        X, _, y = self.prepare_features(df_train)

        tscv = TimeSeriesSplit(n_splits=2)
        best_val_loss_lstm = float('inf')
        best_val_loss_lgb = float('inf')
        best_lstm_state = None
        best_lgb_model = None

        fold = 0
        for train_idx, val_idx in tscv.split(X):
            fold += 1
            print(f"\n=== Fold {fold} ===")

            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # --- LSTM ---
            X_tr_scaled = self.scaler.fit_transform(X_train)
            X_val_scaled = self.scaler.transform(X_val)

            X_tr_seq = self.create_sequences(X_tr_scaled, self.seq_length)
            y_tr_seq = y_train.iloc[self.seq_length-1:].values
            X_val_seq = self.create_sequences(X_val_scaled, self.seq_length)
            y_val_seq = y_val.iloc[self.seq_length-1:].values

            self.build_lstm((self.seq_length, X_train.shape[1]))
            val_loss = self.train_lstm(
                X_tr_seq, y_tr_seq, X_val_seq, y_val_seq,
                epochs=self.lstm_epochs, batch_size=self.batch_size,
                patience=self.lstm_patience
            )

            if val_loss < best_val_loss_lstm:
                best_val_loss_lstm = val_loss
                best_lstm_state = self.lstm_model.state_dict().copy()

            # --- LightGBM ---
            self.train_lgbm(X_train, y_train, X_val, y_val)
            lgb_val_loss = self.lgb_model.best_score['valid_0']['binary_logloss']
            if lgb_val_loss < best_val_loss_lgb:
                best_val_loss_lgb = lgb_val_loss
                best_lgb_model = self.lgb_model

        # Загружаем лучшие модели
        if best_lstm_state is not None:
            self.lstm_model.load_state_dict(best_lstm_state)
            print(f"\nBest LSTM restored, val_loss: {best_val_loss_lstm:.4f}")
        if best_lgb_model is not None:
            self.lgb_model = best_lgb_model
            print(f"Best LightGBM restored, val_logloss: {best_val_loss_lgb:.4f}")

        return self

    # 7. ПРЕДСКАЗАНИЕ
    def predict(self, X: pd.DataFrame, use_ensemble: bool = True) -> np.ndarray:
        """Возвращает вероятности аномалии через forecast_horizon минут."""
        if use_ensemble and self.lgb_model is not None and self.lstm_model is not None:
            lgb_proba = self.lgb_model.predict(X, num_iteration=self.lgb_model.best_iteration)

            X_scaled = self.scaler.transform(X)
            X_seq = self.create_sequences(X_scaled, self.seq_length)

            lstm_proba = np.zeros(len(X))
            if len(X_seq) > 0:
                self.lstm_model.eval()
                with torch.no_grad():
                    X_tensor = torch.FloatTensor(X_seq).to(self.device)
                    lstm_pred = torch.sigmoid(self.lstm_model(X_tensor)).cpu().numpy().flatten()
                    #lstm_pred = self.lstm_model(X_tensor).cpu().numpy().flatten()
                    lstm_proba[-len(lstm_pred):] = lstm_pred
                    lstm_proba[:self.seq_length] = lgb_proba[:self.seq_length]
                    
                    
            lgb_weight = self.config['lgb_weight']
            lstm_weight = self.config['lstm_weight']
            return lgb_weight * lgb_proba + lstm_weight * lstm_proba
            #return 0.6 * lgb_proba + 0.4 * lstm_proba
        elif self.lgb_model is not None:
            return self.lgb_model.predict(X, num_iteration=self.lgb_model.best_iteration)
        else:
            raise ValueError("Model not trained.")
    def train_lgbm_cv(self, df_train, df_val, params):
        X_train, _, y_train = self.prepare_features(df_train)
        X_val, _, y_val = self.prepare_features(df_val)
        
        # Обновляем параметры
        self.lgb_params.update(params)
        
        # Веса классов (обязательно внутри, т.к. зависит от y_train)
        bc = np.bincount(y_train.astype(int))
        class_weights = len(y_train) / (2 * bc)
        weight_map = {i: class_weights[i] for i in range(len(class_weights))}
        sample_weights = y_train.map(weight_map)
        
        train_data = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        self.lgb_model = lgb.train(
            self.lgb_params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=self.lgb_num_boost_round,
            callbacks=[lgb.early_stopping(self.lgb_early_stopping), lgb.log_evaluation(0)]
        )
        return self.lgb_model
    #доработать для lstm
    def train_lstm_cv(self, df_train, df_val, params):
        X_train, _, y_train = self.prepare_features(df_train)
        X_val, _, y_val = self.prepare_features(df_val)
        X, _, y = self.prepare_features(df_train)
        
        self.lstm_params.update(params)

        best_val_loss_lstm = float('inf')
        best_val_loss_lgb = float('inf')
        best_lstm_state = None
        best_lgb_model = None

        X_tr_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)

        X_tr_seq = self.create_sequences(X_tr_scaled, self.seq_length)
        y_tr_seq = y_train.iloc[self.seq_length-1:].values
        X_val_seq = self.create_sequences(X_val_scaled, self.seq_length)
        y_val_seq = y_val.iloc[self.seq_length-1:].values

        self.build_lstm((self.seq_length, X_train.shape[1]))
        val_loss = self.train_lstm(
            X_tr_seq, y_tr_seq, X_val_seq, y_val_seq,
            epochs=self.lstm_epochs, batch_size=self.batch_size,
            patience=self.lstm_patience
        )
        #/////////
        if val_loss < best_val_loss_lstm:
                best_val_loss_lstm = val_loss
                best_lstm_state = self.lstm_model.state_dict().copy()

            # --- LightGBM ---
        self.train_lgbm(X_train, y_train, X_val, y_val)
        lgb_val_loss = self.lgb_model.best_score['valid_0']['binary_logloss']
        if lgb_val_loss < best_val_loss_lgb:
                best_val_loss_lgb = lgb_val_loss
                best_lgb_model = self.lgb_model
        
        # Обновляем параметры
        self.lgb_params.update(params)
        
        # Веса классов (обязательно внутри, т.к. зависит от y_train)
        bc = np.bincount(y_train.astype(int))
        class_weights = len(y_train) / (2 * bc)
        weight_map = {i: class_weights[i] for i in range(len(class_weights))}
        sample_weights = y_train.map(weight_map)
        
        train_data = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        self.lgb_model = lgb.train(
            self.lgb_params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=self.lgb_num_boost_round,
            callbacks=[lgb.early_stopping(self.lgb_early_stopping)]
        )
        return self.lgb_model
        
        
if __name__ == "__main__":
    # Конфигурация (единственное место, где задаются имена колонок и пороги)
    config = {
        'random_state': 19,
        # Имена колонок во входном DataFrame
        'temp_col': 'Камера 2 Температура °С',
        'humid_col': 'Камера 2 Влажность, %',
        'time_col': 'Дата/время',
        # Пороги срабатывания аварии
        'temp_threshold': -7.0,
        'humid_threshold': 80.0,
        # Горизонт предсказания (минуты)
        'forecast_horizon': 5,
        # Признаки
        'features': {'rolling_windows': [5, 15, 30, 60]},
        'lstm_weight': 0.3,               # вес LSTM в ансамбле (было 0.6)
        'lgb_weight': 0.7,               # вес LGBM
        'pred_threshold': 0.5,            # порог вероятности (подберите позже)
        # Параметры LSTM
        'params_lstm':
            {'lstm_pos_weight': 15.0,          # множитель для класса 1 в LSTM
            'lstm_hidden_size1': 64,
            'lstm_hidden_size2': 32,
            'lstm_fc_size': 16,
            'dropout': 0.2,
            'seq_length': 30,
            'lstm_epochs': 100,
            'lstm_patience': 20,
            'batch_size': 32
        },
        # Параметры LightGBM
        'lgb_num_boost_round': 5000,
        'lgb_early_stopping': 100,
        'params_lgbm':
            {'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'verbose': -1,
            'num_threads': 8,
            'lgb_num_leaves': 87,
            'max_depth': 3,
            'min_data_in_leaf': 166,
            'min_sum_hessian_in_leaf': 0.1,
            'lgb_learning_rate': 0.1,
            'feature_fraction': 0.7,
            'bagging_fraction': 0.9,
            'bagging_freq': 5,
            'lambda_l1': 9,
            'lambda_l2': 8,
            'is_unbalance': False
        }
    }

    
    best_params = study.best_trial.params

    # Обновление верхнеуровневых параметров (если они есть в best_params)
    top_keys = ['lgb_weight', 'lstm_weight', 'pred_threshold']
    for key in top_keys:
        if key in best_params:
            config[key] = best_params[key]
    # Обновление параметров LightGBM
    lgbm_dict = config['params_lgbm']
    for key, value in best_params.items():
        if key in lgbm_dict:
            lgbm_dict[key] = value
    # Обновление параметров LSTM
    lstm_dict = config['params_lstm']
    for key, value in best_params.items():
        if key in lstm_dict:
            lstm_dict[key] = value

    # 3. Создаём объект и обучаем
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(pd.concat([train, val]))

    # 4. Тестирование
    test_df = test.copy()

    X_test, _, y_test = predictor.prepare_features(test_df)  # y_test = target (метка за 15 мин)
    pred_proba = predictor.predict(X_test, use_ensemble=False)
    pred_label = (pred_proba > 0.5).astype(int)

    # 5. Оценка
    print("Confusion Matrix:")
    print(classification_report(y_test, pred_label, target_names=['Норма', f'Аномалия через {predictor.forecast_horizon} мин']))
    print(confusion_matrix(y_test, pred_label))

    # 6. Визуализация с Plotly
    # Берём только те строки, для которых были сделаны предсказания
    plot_df = test_df.loc[X_test.index].copy()

    # Устанавливаем временной индекс для корректной оси X
    plot_df.set_index(predictor.time_col, inplace=True)

    # Добавляем предсказанные вероятности и бинарные метки
    plot_df['pred_proba'] = pred_proba
    plot_df['pred_anomaly'] = pred_label
    y_test.index = plot_df.index
    plot_df = pd.concat([plot_df, y_test], axis=1)

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(f'{predictor.temp_col} и аварии',
                                        f'{predictor.humid_col} и аварии'),
                        vertical_spacing=0.1)

    # ----- Температура -----
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df[predictor.temp_col],
                            mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)

    # Реальные аварии (is_failure) – зелёные маркеры
    actual = plot_df[plot_df['target'] == 1]
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual.index, y=actual[predictor.temp_col],
                                mode='markers', name='Реальная авария',
                                marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)

    # Предсказания за 15 минут – **ВСЕ** красные вертикальные линии (без прореживания)
    pred_times = plot_df[plot_df['pred_anomaly'] == 1].index
    for t in pred_times:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)

    # ----- Влажность -----
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df[predictor.humid_col],
                            mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)

    # Реальные аварии на влажности
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual.index, y=actual[predictor.humid_col],
                                mode='markers', name='Реальная авария',
                                marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)

    # Все предсказания для второго графика
    for t in pred_times:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)

    fig.update_layout(height=800,
                    title=f"Мониторинг холодильной камеры: предсказание за {predictor.forecast_horizon} мин")
    fig.show()


=== Fold 1 ===
LSTM Epoch 5/100, Train Loss: 0.3732, Val Loss: 0.5493 (best: 0.5493)
LSTM Epoch 10/100, Train Loss: 0.2687, Val Loss: 1.0178 (best: 0.5493)
LSTM Epoch 15/100, Train Loss: 0.2017, Val Loss: 0.8389 (best: 0.5493)
LSTM Epoch 20/100, Train Loss: 0.1583, Val Loss: 1.3093 (best: 0.5493)
LSTM early stopping at epoch 25, best val_loss: 0.5493
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[272]	valid_0's binary_logloss: 0.278422

=== Fold 2 ===
LSTM Epoch 5/100, Train Loss: 0.2993, Val Loss: 1.1867 (best: 0.9816)
LSTM Epoch 10/100, Train Loss: 0.1729, Val Loss: 3.5342 (best: 0.9816)
LSTM Epoch 15/100, Train Loss: 0.1444, Val Loss: 4.5728 (best: 0.9816)
LSTM Epoch 20/100, Train Loss: 0.1065, Val Loss: 6.1596 (best: 0.9816)
LSTM early stopping at epoch 21, best val_loss: 0.9816
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[77]	valid_0's binary_logloss: 0.175813

Best LSTM restor

In [74]:
def objective(trial):
    # --- Предлагаемые гиперпараметры ---
    config = {
        'random_state': 19,
        # Имена колонок во входном DataFrame
        'temp_col': 'Камера 2 Температура °С',
        'humid_col': 'Камера 2 Влажность, %',
        'time_col': 'Дата/время',
        # Пороги срабатывания аварии
        'temp_threshold': -7.0,
        'humid_threshold': 80.0,
        # Горизонт предсказания (минуты)
        'forecast_horizon': 10,
        # Признаки
        'features': {'rolling_windows': [5, 15, 30, 60]},
        'lstm_weight': trial.suggest_float('lstm_weight', 0.2, 1.0),               # вес LSTM в ансамбле (было 0.6)
        'lgb_weight': trial.suggest_float('lgb_weight', 0.2, 1.0),               # вес LGBM
        'pred_threshold': trial.suggest_float('pred_threshold', 0.3, 1.6),            # порог вероятности (подберите позже)
        # Параметры LSTM
        'params_lstm':
            {'lstm_pos_weight': trial.suggest_int('lstm_pos_weight', 1, 15),          # множитель для класса 1 в LSTM
            'lstm_hidden_size1': trial.suggest_categorical('lstm_hidden_size1', [16, 32, 64, 128]),
            'lstm_hidden_size2': trial.suggest_categorical('lstm_hidden_size2', [16, 32, 64, 128]),
            'lstm_fc_size': trial.suggest_categorical('lstm_fc_size', [16, 32, 64, 128]),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),
            'seq_length': trial.suggest_categorical('seq_length', [10, 20, 30, 40, 50]),
            'lstm_epochs': 100,
            'lstm_patience': 20,
            'batch_size': 32
        },
        # Параметры LightGBM
        'lgb_num_boost_round': 5000,
        'lgb_early_stopping': 100,
        'params_lgbm':
            {'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'verbose': -1,
            'num_threads': 8,
            'lgb_num_leaves': trial.suggest_int('lgb_num_leaves', 15, 255, step=8),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200),
            'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 10.0, log=True),
            'lgb_learning_rate': trial.suggest_float('lgb_learning_rate', 0.01, 0.3, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'lambda_l1': trial.suggest_float('lambda_l1', 0, 10.0),
            'lambda_l2': trial.suggest_float('lambda_l2', 0, 10.0),
            'is_unbalance': False
        }
    }
    
    # --- Обучение ---
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(train)
    
    # --- Оценка ---
    X_val, _, y_val = predictor.prepare_features(val)
    pred_proba = predictor.predict(X_val, use_ensemble=True)
    #pred_label = (pred_proba > 0.5).astype(int)
    #pr_auc = average_precision_score(y_val, pred_proba)
    #prec, rec, thresh = precision_recall_curve(y_val, pred_proba, pos_label=1)
    #f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
    #best_thresh = thresh[np.argmax(f1_scores[:-1])]
    pred_val = (pred_proba > config['pred_threshold']).astype(int)
    f1 = f1_score(y_val, pred_val, pos_label=1)
    return f1

study_f1 = optuna.create_study(direction='maximize', study_name='lgbm_optim')
study_f1.optimize(objective, n_trials=250, show_progress_bar=True)

print("Best trial:")
print(study_f1.best_trial.params)
print("Best F1:", study_f1.best_value)

[I 2026-02-16 22:21:07,520] A new study created in memory with name: lgbm_optim


  0%|          | 0/250 [00:00<?, ?it/s]


=== Fold 1 ===
LSTM Epoch 5/100, Train Loss: 0.4418, Val Loss: 1.3275 (best: 1.0898)
LSTM Epoch 10/100, Train Loss: 0.1948, Val Loss: 1.8947 (best: 1.0898)
LSTM Epoch 15/100, Train Loss: 0.1839, Val Loss: 2.3340 (best: 1.0898)
LSTM Epoch 20/100, Train Loss: 0.0930, Val Loss: 3.3154 (best: 1.0898)
LSTM early stopping at epoch 21, best val_loss: 1.0898
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.71948

=== Fold 2 ===
LSTM Epoch 5/100, Train Loss: 0.3046, Val Loss: 2.4001 (best: 1.2403)
LSTM Epoch 10/100, Train Loss: 0.1989, Val Loss: 3.6484 (best: 1.2403)
LSTM Epoch 15/100, Train Loss: 0.1136, Val Loss: 4.6190 (best: 1.2403)
LSTM Epoch 20/100, Train Loss: 0.0987, Val Loss: 7.0168 (best: 1.2403)
LSTM early stopping at epoch 21, best val_loss: 1.2403
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[73]	valid_0's binary_logloss: 0.258951

Best LSTM restored,

In [75]:
if __name__ == "__main__":
    # Конфигурация (единственное место, где задаются имена колонок и пороги)
    config = {
        'random_state': 19,
        # Имена колонок во входном DataFrame
        'temp_col': 'Камера 2 Температура °С',
        'humid_col': 'Камера 2 Влажность, %',
        'time_col': 'Дата/время',
        # Пороги срабатывания аварии
        'temp_threshold': -7.0,
        'humid_threshold': 80.0,
        # Горизонт предсказания (минуты)
        'forecast_horizon': 5,
        # Признаки
        'features': {'rolling_windows': [5, 15, 30, 60]},
        'lstm_weight': 0.3,               # вес LSTM в ансамбле (было 0.6)
        'lgb_weight': 0.7,               # вес LGBM
        'pred_threshold': 0.5,            # порог вероятности (подберите позже)
        # Параметры LSTM
        'params_lstm':
            {'lstm_pos_weight': 15.0,          # множитель для класса 1 в LSTM
            'lstm_hidden_size1': 64,
            'lstm_hidden_size2': 32,
            'lstm_fc_size': 16,
            'dropout': 0.2,
            'seq_length': 30,
            'lstm_epochs': 100,
            'lstm_patience': 20,
            'batch_size': 32
        },
        # Параметры LightGBM
        'lgb_num_boost_round': 5000,
        'lgb_early_stopping': 100,
        'params_lgbm':
            {'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'verbose': -1,
            'num_threads': 8,
            'lgb_num_leaves': 87,
            'max_depth': 3,
            'min_data_in_leaf': 166,
            'min_sum_hessian_in_leaf': 0.1,
            'lgb_learning_rate': 0.1,
            'feature_fraction': 0.7,
            'bagging_fraction': 0.9,
            'bagging_freq': 5,
            'lambda_l1': 9,
            'lambda_l2': 8,
            'is_unbalance': False
        }
    }

    
    best_params = study_f1.best_trial.params

    # Обновление верхнеуровневых параметров (если они есть в best_params)
    top_keys = ['lgb_weight', 'lstm_weight', 'pred_threshold']
    for key in top_keys:
        if key in best_params:
            config[key] = best_params[key]
    # Обновление параметров LightGBM
    lgbm_dict = config['params_lgbm']
    for key, value in best_params.items():
        if key in lgbm_dict:
            lgbm_dict[key] = value
    # Обновление параметров LSTM
    lstm_dict = config['params_lstm']
    for key, value in best_params.items():
        if key in lstm_dict:
            lstm_dict[key] = value

    # 3. Создаём объект и обучаем
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(pd.concat([train, val]))

    # 4. Тестирование
    test_df = test.copy()

    X_test, _, y_test = predictor.prepare_features(test_df)  # y_test = target (метка за 15 мин)
    pred_proba = predictor.predict(X_test, use_ensemble=False)
    pred_label = (pred_proba > 0.5).astype(int)

    # 5. Оценка
    print("Confusion Matrix:")
    print(classification_report(y_test, pred_label, target_names=['Норма', f'Аномалия через {predictor.forecast_horizon} мин']))
    print(confusion_matrix(y_test, pred_label))

    # 6. Визуализация с Plotly
    # Берём только те строки, для которых были сделаны предсказания
    plot_df = test_df.loc[X_test.index].copy()

    # Устанавливаем временной индекс для корректной оси X
    plot_df.set_index(predictor.time_col, inplace=True)

    # Добавляем предсказанные вероятности и бинарные метки
    plot_df['pred_proba'] = pred_proba
    plot_df['pred_anomaly'] = pred_label
    y_test.index = plot_df.index
    plot_df = pd.concat([plot_df, y_test], axis=1)

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(f'{predictor.temp_col} и аварии',
                                        f'{predictor.humid_col} и аварии'),
                        vertical_spacing=0.1)

    # ----- Температура -----
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df[predictor.temp_col],
                            mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)

    # Реальные аварии (is_failure) – зелёные маркеры
    actual = plot_df[plot_df['target'] == 1]
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual.index, y=actual[predictor.temp_col],
                                mode='markers', name='Реальная авария',
                                marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)

    # Предсказания за 15 минут – **ВСЕ** красные вертикальные линии (без прореживания)
    pred_times = plot_df[plot_df['pred_anomaly'] == 1].index
    for t in pred_times:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)

    # ----- Влажность -----
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df[predictor.humid_col],
                            mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)

    # Реальные аварии на влажности
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual.index, y=actual[predictor.humid_col],
                                mode='markers', name='Реальная авария',
                                marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)

    # Все предсказания для второго графика
    for t in pred_times:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)

    fig.update_layout(height=800,
                    title=f"Мониторинг холодильной камеры: предсказание за {predictor.forecast_horizon} мин")
    fig.show()


=== Fold 1 ===
LSTM Epoch 5/100, Train Loss: 0.2569, Val Loss: 0.4269 (best: 0.3968)
LSTM Epoch 10/100, Train Loss: 0.1706, Val Loss: 0.6138 (best: 0.3968)
LSTM Epoch 15/100, Train Loss: 0.1503, Val Loss: 0.7393 (best: 0.3968)
LSTM Epoch 20/100, Train Loss: 0.1146, Val Loss: 0.7279 (best: 0.3968)
LSTM early stopping at epoch 23, best val_loss: 0.3968
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[238]	valid_0's binary_logloss: 0.289764

=== Fold 2 ===
LSTM Epoch 5/100, Train Loss: 0.1935, Val Loss: 0.8097 (best: 0.6262)
LSTM Epoch 10/100, Train Loss: 0.1201, Val Loss: 1.4469 (best: 0.6262)
LSTM Epoch 15/100, Train Loss: 0.0859, Val Loss: 1.2840 (best: 0.6262)
LSTM Epoch 20/100, Train Loss: 0.0704, Val Loss: 2.7058 (best: 0.6262)
LSTM early stopping at epoch 24, best val_loss: 0.6262
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[147]	valid_0's binary_logloss: 0.167472

Best LSTM resto

In [76]:
def objective(trial):
    # --- Предлагаемые гиперпараметры ---
    config = {
        'random_state': 19,
        # Имена колонок во входном DataFrame
        'temp_col': 'Камера 2 Температура °С',
        'humid_col': 'Камера 2 Влажность, %',
        'time_col': 'Дата/время',
        # Пороги срабатывания аварии
        'temp_threshold': -7.0,
        'humid_threshold': 80.0,
        # Горизонт предсказания (минуты)
        'forecast_horizon': 10,
        # Признаки
        'features': {'rolling_windows': [5, 15, 30, 60]},
        'lstm_weight': trial.suggest_float('lstm_weight', 0.2, 1.0),               # вес LSTM в ансамбле (было 0.6)
        'lgb_weight': trial.suggest_float('lgb_weight', 0.2, 1.0),               # вес LGBM
        'pred_threshold': trial.suggest_float('pred_threshold', 0.3, 1.6),            # порог вероятности (подберите позже)
        # Параметры LSTM
        'params_lstm':
            {'lstm_pos_weight': trial.suggest_int('lstm_pos_weight', 1, 15),          # множитель для класса 1 в LSTM
            'lstm_hidden_size1': trial.suggest_categorical('lstm_hidden_size1', [16, 32, 64, 128]),
            'lstm_hidden_size2': trial.suggest_categorical('lstm_hidden_size2', [16, 32, 64, 128]),
            'lstm_fc_size': trial.suggest_categorical('lstm_fc_size', [16, 32, 64, 128]),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),
            'seq_length': trial.suggest_categorical('seq_length', [10, 20, 30, 40, 50]),
            'lstm_epochs': 100,
            'lstm_patience': 20,
            'batch_size': 32
        },
        # Параметры LightGBM
        'lgb_num_boost_round': 5000,
        'lgb_early_stopping': 100,
        'params_lgbm':
            {'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'verbose': -1,
            'num_threads': 8,
            'lgb_num_leaves': trial.suggest_int('lgb_num_leaves', 15, 255, step=8),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200),
            'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 10.0, log=True),
            'lgb_learning_rate': trial.suggest_float('lgb_learning_rate', 0.01, 0.3, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'lambda_l1': trial.suggest_float('lambda_l1', 0, 10.0),
            'lambda_l2': trial.suggest_float('lambda_l2', 0, 10.0),
            'is_unbalance': False
        }
    }
    
    # --- Обучение ---
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(train)
    
    # --- Оценка ---
    X_val, _, y_val = predictor.prepare_features(val)
    pred_proba = predictor.predict(X_val, use_ensemble=True)
    #pred_label = (pred_proba > 0.5).astype(int)
    pr_auc = average_precision_score(y_val, pred_proba)
    #prec, rec, thresh = precision_recall_curve(y_val, pred_proba, pos_label=1)
    #f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
    #best_thresh = thresh[np.argmax(f1_scores[:-1])]
    #pred_val = (pred_proba > config['pred_threshold']).astype(int)
    #f1 = f1_score(y_val, pred_val, pos_label=1)
    return pr_auc

study = optuna.create_study(direction='maximize', study_name='lgbm_optim')
study.optimize(objective, n_trials=250, show_progress_bar=True)

print("Best trial:")
print(study.best_trial.params)
print("Best F1:", study.best_value)

[I 2026-02-16 23:51:08,921] A new study created in memory with name: lgbm_optim


  0%|          | 0/250 [00:00<?, ?it/s]


=== Fold 1 ===
LSTM Epoch 5/100, Train Loss: 0.2795, Val Loss: 1.1291 (best: 0.8111)
LSTM Epoch 10/100, Train Loss: 0.1840, Val Loss: 1.5095 (best: 0.8111)
LSTM Epoch 15/100, Train Loss: 0.1497, Val Loss: 1.3694 (best: 0.8111)
LSTM Epoch 20/100, Train Loss: 0.1032, Val Loss: 1.5418 (best: 0.8111)
LSTM early stopping at epoch 21, best val_loss: 0.8111
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.715631

=== Fold 2 ===
LSTM Epoch 5/100, Train Loss: 0.2708, Val Loss: 1.8457 (best: 0.9038)
LSTM Epoch 10/100, Train Loss: 0.1272, Val Loss: 2.1404 (best: 0.9038)
LSTM Epoch 15/100, Train Loss: 0.0949, Val Loss: 3.1772 (best: 0.9038)
LSTM Epoch 20/100, Train Loss: 0.0806, Val Loss: 3.2682 (best: 0.9038)
LSTM early stopping at epoch 21, best val_loss: 0.9038
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[136]	valid_0's binary_logloss: 0.252623

Best LSTM restore

In [77]:
if __name__ == "__main__":
    # Конфигурация (единственное место, где задаются имена колонок и пороги)
    config = {
        'random_state': 19,
        # Имена колонок во входном DataFrame
        'temp_col': 'Камера 2 Температура °С',
        'humid_col': 'Камера 2 Влажность, %',
        'time_col': 'Дата/время',
        # Пороги срабатывания аварии
        'temp_threshold': -7.0,
        'humid_threshold': 80.0,
        # Горизонт предсказания (минуты)
        'forecast_horizon': 5,
        # Признаки
        'features': {'rolling_windows': [5, 15, 30, 60]},
        'lstm_weight': 0.3,               # вес LSTM в ансамбле (было 0.6)
        'lgb_weight': 0.7,               # вес LGBM
        'pred_threshold': 0.5,            # порог вероятности (подберите позже)
        # Параметры LSTM
        'params_lstm':
            {'lstm_pos_weight': 15.0,          # множитель для класса 1 в LSTM
            'lstm_hidden_size1': 64,
            'lstm_hidden_size2': 32,
            'lstm_fc_size': 16,
            'dropout': 0.2,
            'seq_length': 30,
            'lstm_epochs': 100,
            'lstm_patience': 20,
            'batch_size': 32
        },
        # Параметры LightGBM
        'lgb_num_boost_round': 5000,
        'lgb_early_stopping': 100,
        'params_lgbm':
            {'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'verbose': -1,
            'num_threads': 8,
            'lgb_num_leaves': 87,
            'max_depth': 3,
            'min_data_in_leaf': 166,
            'min_sum_hessian_in_leaf': 0.1,
            'lgb_learning_rate': 0.1,
            'feature_fraction': 0.7,
            'bagging_fraction': 0.9,
            'bagging_freq': 5,
            'lambda_l1': 9,
            'lambda_l2': 8,
            'is_unbalance': False
        }
    }

    
    best_params = study.best_trial.params

    # Обновление верхнеуровневых параметров (если они есть в best_params)
    top_keys = ['lgb_weight', 'lstm_weight', 'pred_threshold']
    for key in top_keys:
        if key in best_params:
            config[key] = best_params[key]
    # Обновление параметров LightGBM
    lgbm_dict = config['params_lgbm']
    for key, value in best_params.items():
        if key in lgbm_dict:
            lgbm_dict[key] = value
    # Обновление параметров LSTM
    lstm_dict = config['params_lstm']
    for key, value in best_params.items():
        if key in lstm_dict:
            lstm_dict[key] = value

    # 3. Создаём объект и обучаем
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(pd.concat([train, val]))

    # 4. Тестирование
    test_df = test.copy()

    X_test, _, y_test = predictor.prepare_features(test_df)  # y_test = target (метка за 15 мин)
    pred_proba = predictor.predict(X_test, use_ensemble=False)
    pred_label = (pred_proba > 0.5).astype(int)

    # 5. Оценка
    print("Confusion Matrix:")
    print(classification_report(y_test, pred_label, target_names=['Норма', f'Аномалия через {predictor.forecast_horizon} мин']))
    print(confusion_matrix(y_test, pred_label))

    # 6. Визуализация с Plotly
    # Берём только те строки, для которых были сделаны предсказания
    plot_df = test_df.loc[X_test.index].copy()

    # Устанавливаем временной индекс для корректной оси X
    plot_df.set_index(predictor.time_col, inplace=True)

    # Добавляем предсказанные вероятности и бинарные метки
    plot_df['pred_proba'] = pred_proba
    plot_df['pred_anomaly'] = pred_label
    y_test.index = plot_df.index
    plot_df = pd.concat([plot_df, y_test], axis=1)

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(f'{predictor.temp_col} и аварии',
                                        f'{predictor.humid_col} и аварии'),
                        vertical_spacing=0.1)

    # ----- Температура -----
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df[predictor.temp_col],
                            mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)

    # Реальные аварии (is_failure) – зелёные маркеры
    actual = plot_df[plot_df['target'] == 1]
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual.index, y=actual[predictor.temp_col],
                                mode='markers', name='Реальная авария',
                                marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)

    # Предсказания за 15 минут – **ВСЕ** красные вертикальные линии (без прореживания)
    pred_times = plot_df[plot_df['pred_anomaly'] == 1].index
    for t in pred_times:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)

    # ----- Влажность -----
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df[predictor.humid_col],
                            mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)

    # Реальные аварии на влажности
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual.index, y=actual[predictor.humid_col],
                                mode='markers', name='Реальная авария',
                                marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)

    # Все предсказания для второго графика
    for t in pred_times:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)

    fig.update_layout(height=800,
                    title=f"Мониторинг холодильной камеры: предсказание за {predictor.forecast_horizon} мин")
    fig.show()


=== Fold 1 ===
LSTM Epoch 5/100, Train Loss: 0.1666, Val Loss: 0.4156 (best: 0.3124)
LSTM Epoch 10/100, Train Loss: 0.1113, Val Loss: 0.3746 (best: 0.3124)
LSTM Epoch 15/100, Train Loss: 0.0710, Val Loss: 0.5203 (best: 0.3124)
LSTM Epoch 20/100, Train Loss: 0.0678, Val Loss: 0.6079 (best: 0.3124)
LSTM early stopping at epoch 21, best val_loss: 0.3124
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[154]	valid_0's binary_logloss: 0.26133

=== Fold 2 ===
LSTM Epoch 5/100, Train Loss: 0.1390, Val Loss: 0.7516 (best: 0.5167)
LSTM Epoch 10/100, Train Loss: 0.0851, Val Loss: 0.8707 (best: 0.5167)
LSTM Epoch 15/100, Train Loss: 0.0665, Val Loss: 1.4601 (best: 0.5167)
LSTM Epoch 20/100, Train Loss: 0.0422, Val Loss: 1.6004 (best: 0.5167)
LSTM early stopping at epoch 23, best val_loss: 0.5167
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[119]	valid_0's binary_logloss: 0.167635

Best LSTM restor

In [ ]:
def objective_lgbm(trial):
    # --- Предлагаемые гиперпараметры ---
    config = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'verbose': -1,
        'num_threads': 8,
        'num_leaves': trial.suggest_int('num_leaves', 15, 255, step=8),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200),
        'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1': trial.suggest_float('lambda_l1', 0, 10.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0, 10.0),
        'is_unbalance': False
    }
    
    # --- Обучение ---
    predictor = AnomalyPredictor(config)
    predictor.train_lgbm_cv(train, val, params)
    
    # --- Оценка ---
    X_val, _, y_val = predictor.prepare_features(val)
    pred_proba = predictor.predict(X_val, use_ensemble=False)
    pred_label = (pred_proba > 0.5).astype(int)
    pr_auc = average_precision_score(y_val, pred_proba)
    #prec, rec, thresh = precision_recall_curve(y_val, pred_proba, pos_label=1)
    #f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
    #best_thresh = thresh[np.argmax(f1_scores[:-1])]
    #pred_val = (pred_proba > best_thresh).astype(int)
    #f1 = f1_score(y_val, pred_val, pos_label=1)
    return pr_auc

study = optuna.create_study(direction='maximize', study_name='lgbm_optim')
study.optimize(objective_lgbm, n_trials=150, show_progress_bar=True)

print("Best trial:")
print(study.best_trial.params)
print("Best F1:", study.best_value)

[I 2026-02-13 18:05:18,267] A new study created in memory with name: lgbm_optim


  0%|          | 0/150 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[291]	valid_0's binary_logloss: 0.198982
[I 2026-02-13 18:05:20,771] Trial 0 finished with value: 0.5561504576093942 and parameters: {'num_leaves': 207, 'max_depth': 14, 'min_data_in_leaf': 139, 'min_sum_hessian_in_leaf': 2.4359151349739427, 'learning_rate': 0.05840313610438349, 'feature_fraction': 0.6917039866181947, 'bagging_fraction': 0.7316928064705723, 'bagging_freq': 4, 'lambda_l1': 1.4774308776197442, 'lambda_l2': 7.094161732097541}. Best is trial 0 with value: 0.5561504576093942.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1114]	valid_0's binary_logloss: 0.216285
[I 2026-02-13 18:05:23,301] Trial 1 finished with value: 0.5231244141769821 and parameters: {'num_leaves': 39, 'max_depth': 3, 'min_data_in_leaf': 78, 'min_sum_hessian_in_leaf': 0.23710950915407242, 'learning_rate': 0.05139805256471802, 'feature_fraction': 0.683407738084

In [ ]:
def objective_lgbm(trial):
    # --- Предлагаемые гиперпараметры ---
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'verbose': -1,
        'num_threads': 8,
        'num_leaves': trial.suggest_int('num_leaves', 15, 255, step=8),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200),
        'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1': trial.suggest_float('lambda_l1', 0, 10.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0, 10.0),
        'is_unbalance': True
    }
    
    # --- Обучение ---
    predictor = AnomalyPredictor(config)
    predictor.train_lgbm_cv(train_df, val_df, params)
    
    # --- Оценка ---
    X_val, _, y_val = predictor.prepare_features(val_df)
    pred_proba = predictor.predict(X_val, use_ensemble=False)
    pred_label = (pred_proba > 0.5).astype(int)
    f1 = f1_score(y_val, pred_label, pos_label=1)
    return f1

study = optuna.create_study(direction='maximize', study_name='lgbm_optim')
study.optimize(objective_lgbm, n_trials=150, show_progress_bar=True)

print("Best trial:")
print(study.best_trial.params)
print("Best F1:", study.best_value)

In [ ]:
def objective_lstm(trial):
    params = {
        'seq_length': trial.suggest_int('seq_length', 30, 120, step=10),
        'lstm_epochs': trial.suggest_int('lstm_epochs', 20, 100),
        'lstm_patience': trial.suggest_int('lstm_patience', 5, 20),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
        'lstm_pos_weight': trial.suggest_float('lstm_pos_weight', 1.0, 20.0, log=True),
        'lstm_lr': trial.suggest_float('lstm_lr', 1e-4, 1e-2, log=True),
        'lstm_dropout': trial.suggest_float('lstm_dropout', 0.1, 0.5),
        'lstm_units': trial.suggest_int('lstm_units', 32, 128, step=16)
    }
    # Обновите конфиг, создайте предсказатель, обучите LSTM (с фиксированным LGBM или без него)
    # Оцените F1 на валидации
        # --- Обучение ---
    predictor = AnomalyPredictor(config)
    predictor.train_lstm_cv(train_df, val_df, params)  # доработаем метод
    
    # --- Оценка ---
    X_val, _, y_val = predictor.prepare_features(val_df)
    pred_proba = predictor.predict(X_val, use_ensemble=False)
    pred_label = (pred_proba > 0.5).astype(int)
    f1 = f1_score(y_val, pred_label, pos_label=1)
    return f1

study = optuna.create_study(direction='maximize', study_name='lgbm_optim')
study.optimize(objective_lgbm, n_trials=50)

print("Best trial:")
print(study.best_trial.params)
print("Best F1:", study.best_value)

In [84]:
TEMP_COL = 'Камера 2 Температура °С'
HUMID_COL = 'Камера 2 Влажность, %'
TIME_COL = 'Дата/время'
TEMP_THRESHOLD = -7.0
HUMID_THRESHOLD = 80.0
FORECAST_HORIZON = 5

class AnomalyPredictor:
    """Ансамблевая модель для предсказания аварий (LGBM + LSTM на PyTorch)"""
    
    def __init__(self, config: dict):
        self.config = config
        self.lgb_model = None
        self.lstm_model = None
        self.scaler = StandardScaler()
        self.feature_cols = None
        self.device = torch.device('cuda')
    
    def prepare_features(self, df_pf: pd.DataFrame):
        """Подготовка фичей для разных моделей"""
        feature_engineer = FeatureEngineer()
        data = feature_engineer.create_features(df_pf)
        
        labeling = LabelGenerator()
        data = labeling.create_labels(data)
        data = data.rename(columns={TEMP_COL:'temp' , HUMID_COL: 'hum'})
        # Выбираем только числовые колонки
        numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
        exclude_cols = ['target', 'is_failure', 'time_to_failure', 'ttf_log', 'timestamp']
        self.feature_cols = [col for col in numeric_cols if col not in exclude_cols]
        # Для совместимости: создаём target = is_failure (сдвиг будет позже, здесь просто метка)
        #if 'is_failure' in data.columns:
        #    data['target'] = data['is_failure']
        return data[self.feature_cols], data['is_failure'], data['target'] if 'target' in data.columns else None
    
    def train_lgbm(self, X_train, y_train, X_val, y_val):
        class_weights = len(y_train) / (2 * np.bincount(y_train.astype(int)))
        weight_map = {i: class_weights[i] for i in range(len(class_weights))}
        sample_weights = y_train.map(weight_map)
        
        train_data = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': 0,
            'num_threads': 4,
            'min_data_in_leaf': 20
        }
        
        self.lgb_model = lgb.train(
            params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(50)]
        )
        return self.lgb_model
    
    def build_lstm(self, input_shape):
        seq_length, n_features = input_shape
        
        class LSTMPredictor(nn.Module):
            def __init__(self, seq_len, feat_dim):
                super().__init__()
                self.lstm1 = nn.LSTM(feat_dim, 64, batch_first=True, dropout=0.2)
                self.lstm2 = nn.LSTM(64, 32, batch_first=True, dropout=0.2)
                self.fc1 = nn.Linear(32, 16)
                self.relu = nn.ReLU()
                self.dropout = nn.Dropout(0.2)
                self.fc2 = nn.Linear(16, 1)
                self.sigmoid = nn.Sigmoid()
                
            def forward(self, x):
                out, _ = self.lstm1(x)
                out, _ = self.lstm2(out)
                out = out[:, -1, :]
                out = self.fc1(out)
                out = self.relu(out)
                out = self.dropout(out)
                out = self.fc2(out)
                out = self.sigmoid(out)
                return out.squeeze(1)
        
        self.lstm_model = LSTMPredictor(seq_length, n_features).to(self.device)
        return self.lstm_model
    
    def create_sequences(self, data, seq_length):
        sequences = []
        for i in range(len(data) - seq_length + 1):
            sequences.append(data[i:i+seq_length])
        return np.array(sequences)
    
    def train_lstm(self, X_train_seq, y_train_seq, X_val_seq, y_val_seq, epochs=None, batch_size=32):
        
        # Если epochs не передан, берём из конфига
        if epochs is None:
            epochs = self.config.get('lstm_epochs', 20)
        patience = self.config.get('lstm_patience', 5)  # сколько эпох ждать улучшения
        criterion = nn.BCELoss()
        optimizer = optim.Adam(self.lstm_model.parameters(), lr=0.001)
        
        train_dataset = TensorDataset(torch.FloatTensor(X_train_seq).to(self.device),
                                      torch.FloatTensor(y_train_seq).to(self.device))
        val_dataset = TensorDataset(torch.FloatTensor(X_val_seq).to(self.device),
                                    torch.FloatTensor(y_val_seq).to(self.device))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        best_val_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        
        for epoch in range(epochs):
            self.lstm_model.train()
            train_loss = 0.0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = self.lstm_model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * X_batch.size(0)
            train_loss /= len(train_loader.dataset)
            
            # Валидация
            self.lstm_model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    outputs = self.lstm_model(X_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item() * X_batch.size(0)
            val_loss /= len(val_loader.dataset)
            
            # --- Save best model ---
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = self.lstm_model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}, best val_loss: {best_val_loss:.4f}")
                    break

            if (epoch + 1) % 5 == 0:
                print(f"LSTM Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f} (best: {best_val_loss:.4f})")
                
        # Load the best model weights from memory
        if best_model_state is not None:
            self.lstm_model.load_state_dict(best_model_state)
            print(f"Loaded best LSTM model with validation loss: {best_val_loss:.4f}")
    
    def train_ensemble(self, df_train: pd.DataFrame):
        X, _, y = self.prepare_features(df_train)
        tscv = TimeSeriesSplit(n_splits=3)
        for train_idx, val_idx in tscv.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            # Масштабирование
            X_train_scaled = self.scaler.fit_transform(X_train)
            X_val_scaled = self.scaler.transform(X_val)
            
            seq_length = self.config.get('seq_length', 60)
            lstm_epochs = self.config.get('lstm_epochs', 20)
            X_train_seq = self.create_sequences(X_train_scaled, seq_length)
            y_train_seq = y_train.iloc[seq_length-1:].values
            X_val_seq = self.create_sequences(X_val_scaled, seq_length)
            y_val_seq = y_val.iloc[seq_length-1:].values
            
            # LSTM
            self.build_lstm((seq_length, X_train.shape[1]))
            self.train_lstm(X_train_seq, y_train_seq, X_val_seq, y_val_seq, epochs=10)
            
            # LGBM
            self.train_lgbm(X_train, y_train, X_val, y_val)
            #break
        return self
    
    def predict(self, X: pd.DataFrame, use_ensemble: bool = True):
        if use_ensemble and self.lgb_model is not None and self.lstm_model is not None:
            lgb_proba = self.lgb_model.predict(X)
            
            X_scaled = self.scaler.transform(X)
            X_seq = self.create_sequences(X_scaled, 60)
            
            lstm_proba = np.zeros(len(X))
            if len(X_seq) > 0:
                self.lstm_model.eval()
                with torch.no_grad():
                    X_tensor = torch.FloatTensor(X_seq).to(self.device)
                    lstm_pred = self.lstm_model(X_tensor).cpu().numpy().flatten()
                    lstm_proba[-len(lstm_pred):] = lstm_pred
                    lstm_proba[:60] = lgb_proba[:60]
            
            ensemble_proba = 0.6 * lgb_proba# + 0.4 * lstm_proba
            return ensemble_proba
        elif self.lgb_model is not None:
            return self.lgb_model.predict(X)
        else:
            raise ValueError("Модель не обучена")

# --- 4. Основной скрипт ---
if __name__ == "__main__":
    # Конфигурация модели
    config = {
        'temp_col': 'Камера 2 Температура °С',
        'humid_col': 'Камера 2 Влажность, %',
        'temp_threshold': -7.0,
        'humid_threshold': 80.0,
        'forecast_horizon': 5,
        'features': {
            'rolling_windows': [5, 15, 30, 60]
        },
        # Параметры LSTM
        'seq_length': 60,
        'lstm_epochs': 30,
        'lstm_patience': 7,
        # Параметры LGBM (можно вынести сюда же)
        'lgb_num_leaves': 31,
        'lgb_learning_rate': 0.05,
        'lgb_num_boost_round': 1000,
        'lgb_early_stopping': 50
    }

    # Инициализация и обучение
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(train)
    # --- Подготовка тестовой выборки (последние 2000 точек) ---
    test_df = test.copy()
    #test_df['is_failure'] = ((test_df['Камера 2 Температура °С'] >= -7) | (test_df['Камера 2 Влажность, %'] >= 80)).astype(int)
    #labeling = LabelGenerator()
    #test_df = labeling.create_labels(test_df)
    X_test, v, y_test = predictor.prepare_features(test_df)  # y_test нам не нужен для predict
    #y_test = test_df.loc[X_test.index, 'target']  # реальные метки
    # Получаем вероятности ансамбля
    pred_proba = predictor.predict(X_test, use_ensemble=True)
    pred_label = (pred_proba > 0.5).astype(int)

    # --- Визуализация с Plotly ---
    # Объединяем в один DataFrame для удобства
    plot_df = test_df.copy()
    #plot_df.set_index(TIME_COL, inplace=True)
    plot_df['pred_proba'] = pred_proba
    plot_df['pred_anomaly'] = pred_label
    plot_df = plot_df
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('Температура и аномалии', 'Влажность и аномалии'),
                        vertical_spacing=0.1)

    # Температура
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Камера 2 Температура °С'],
                             mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)
    # Фактические аномалии (is_failure)
    actual_failures = plot_df[plot_df['is_failure'] == 1]
    fig.add_trace(go.Scatter(x=actual_failures.index, y=actual_failures['Камера 2 Температура °С'],
                             mode='markers', name='Реальная авария',
                             marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)
    # Предсказанные аномалии (красные вертикальные линии)
    pred_failures = plot_df[plot_df['pred_anomaly'] == 1]
    for t in pred_failures.index:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)
    # Влажность
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Камера 2 Влажность, %'],
                             mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)
    actual_failures_h = plot_df[plot_df['target'] == 1]
    fig.add_trace(go.Scatter(x=actual_failures_h.index, y=actual_failures_h['Камера 2 Влажность, %'],
                             mode='markers', name='Реальная авария',
                             marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)
    for t in pred_failures.index:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)
    fig.update_layout(height=800, title_text="Мониторинг холодильной камеры: фактические и предсказанные аварии")
    fig.show()

    # --- Дополнительно: метрики на тесте ---
    from sklearn.metrics import classification_report, confusion_matrix
    print("\n=== Отчёт по классификации на тестовой выборке ===")
    print(classification_report(y_test, pred_label, target_names=['Норма', 'Аномалия']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred_label))


LSTM Epoch 5/10, Train Loss: 0.0998, Val Loss: 0.2189 (best: 0.1521)
Early stopping at epoch 10, best val_loss: 0.1521
Loaded best LSTM model with validation loss: 0.1521
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[88]	valid_0's binary_logloss: 0.276306
LSTM Epoch 5/10, Train Loss: 0.0755, Val Loss: 0.0862 (best: 0.0756)
Early stopping at epoch 10, best val_loss: 0.0756
Loaded best LSTM model with validation loss: 0.0756
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[181]	valid_0's binary_logloss: 0.0928726
LSTM Epoch 5/10, Train Loss: 0.0677, Val Loss: 0.3037 (best: 0.2838)
LSTM Epoch 10/10, Train Loss: 0.0401, Val Loss: 0.4377 (best: 0.2713)
Loaded best LSTM model with validation loss: 0.2713
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	valid_0's binary_logloss: 0.231264



=== Отчёт по классификации на тестовой выборке ===
              precision    recall  f1-score   support

       Норма       0.94      1.00      0.97       930
    Аномалия       0.73      0.16      0.26        70

    accuracy                           0.94      1000
   macro avg       0.84      0.58      0.61      1000
weighted avg       0.93      0.94      0.92      1000

Confusion Matrix:
[[926   4]
 [ 59  11]]


In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('Температура и аномалии', 'Влажность и аномалии'),
                        vertical_spacing=0.1)

    # Температура
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Камера 2 Температура °С'],
                             mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)
    # Фактические аномалии (is_failure)
actual_failures = plot_df[plot_df['is_failure'] == 1]
fig.add_trace(go.Scatter(x=actual_failures.index, y=actual_failures['Камера 2 Температура °С'],
                             mode='markers', name='Реальная авария',
                             marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)
    # Предсказанные аномалии (красные вертикальные линии)
pred_failures = plot_df[plot_df['pred_anomaly'] == 1]
for t in pred_failures.index:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)
    # Влажность
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Камера 2 Влажность, %'],
                             mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)
actual_failures_h = plot_df[plot_df['is_failure'] == 1]
fig.add_trace(go.Scatter(x=actual_failures_h.index, y=actual_failures_h['Камера 2 Влажность, %'],
                             mode='markers', name='Реальная авария',
                             marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)
for t in pred_failures.index:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)
fig.update_layout(height=800, title_text="Мониторинг холодильной камеры: фактические и предсказанные аварии")
fig.show()

In [ ]:
TEMP_COL = 'Камера 2 Температура °С'
HUMID_COL = 'Камера 2 Влажность, %'
TEMP_THRESHOLD = -7.0
HUMID_THRESHOLD = 80.0
FORECAST_HORIZON = 15
# Формируем датафрейм для визуализации
plot_df = test_df.loc[X_test[TIME_COL]].copy()
plot_df['pred_proba'] = pred_proba
plot_df['pred_anomaly_15min'] = pred_label

# Создаём фигуру с двумя подграфиками
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=(f'{TEMP_COL} и аварии', f'{HUMID_COL} и аварии'),
                    vertical_spacing=0.1)

# ----- Температура -----
fig.add_trace(go.Scatter(x=plot_df[TIME_COL], y=plot_df[TEMP_COL],
                         mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)

# Реальные аварии (зелёные маркеры)
actual_failures = plot_df[plot_df['is_failure'] == 1]
fig.add_trace(go.Scatter(x=actual_failures[TIME_COL], y=actual_failures[TEMP_COL],
                         mode='markers', name='Реальная авария',
                         marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)

# Предсказания за 15 минут (красные вертикальные линии)
pred_times = plot_df[plot_df['pred_anomaly_15min'] == 1][TIME_COL]
for t in pred_times:
    fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)

# ----- Влажность -----
fig.add_trace(go.Scatter(x=plot_df[TIME_COL], y=plot_df[HUMID_COL],
                         mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)

# Реальные аварии на графике влажности
fig.add_trace(go.Scatter(x=actual_failures[TIME_COL], y=actual_failures[HUMID_COL],
                         mode='markers', name='Реальная авария',
                         marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)

# Красные линии для предсказаний
for t in pred_times:
    fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)

# Общие настройки
fig.update_layout(
    height=800,
    title=f"Мониторинг холодильной камеры: реальные аварии и предсказания за {FORECAST_HORIZON} мин",
    hovermode='x unified'
)
fig.show()

In [ ]:
# Инициализация и обучение
    predictor = AnomalyPredictor(config)
    predictor.train_ensemble(df)

    # --- Подготовка тестовой выборки (последние 2000 точек) ---
    test_df = df.iloc[-2000:].copy()
    X_test, _ = predictor.prepare_features(test_df)  # y_test нам не нужен для predict
    y_test = test_df.loc[X_test.index, 'is_failure']  # реальные метки

    # Получаем вероятности ансамбля
    pred_proba = predictor.predict(X_test, use_ensemble=True)
    pred_label = (pred_proba > 0.5).astype(int)

    # --- Визуализация с Plotly ---
    # Объединяем в один DataFrame для удобства
    plot_df = test_df.loc[X_test.index].copy()
    plot_df['pred_proba'] = pred_proba
    plot_df['pred_anomaly'] = pred_label

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('Температура и аномалии', 'Влажность и аномалии'),
                        vertical_spacing=0.1)

    # Температура
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['temperature'],
                             mode='lines', name='Температура', line=dict(color='blue')), row=1, col=1)
    # Фактические аномалии (is_failure)
    actual_failures = plot_df[plot_df['is_failure'] == 1]
    fig.add_trace(go.Scatter(x=actual_failures.index, y=actual_failures['temperature'],
                             mode='markers', name='Реальная авария',
                             marker=dict(color='green', size=6, symbol='circle')), row=1, col=1)
    # Предсказанные аномалии (красные вертикальные линии)
    pred_failures = plot_df[plot_df['pred_anomaly'] == 1]
    for t in pred_failures.index:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=1, col=1)

    # Влажность
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['humidity'],
                             mode='lines', name='Влажность', line=dict(color='orange')), row=2, col=1)
    actual_failures_h = plot_df[plot_df['is_failure'] == 1]
    fig.add_trace(go.Scatter(x=actual_failures_h.index, y=actual_failures_h['humidity'],
                             mode='markers', name='Реальная авария',
                             marker=dict(color='green', size=6, symbol='circle')), row=2, col=1)
    for t in pred_failures.index:
        fig.add_vline(x=t, line_width=1, line_color='red', opacity=0.3, row=2, col=1)

    fig.update_layout(height=800, title_text="Мониторинг холодильной камеры: фактические и предсказанные аварии")
    fig.show()

    # --- Дополнительно: метрики на тесте ---
    from sklearn.metrics import classification_report, confusion_matrix
    print("\n=== Отчёт по классификации на тестовой выборке ===")
    print(classification_report(y_test, pred_label, target_names=['Норма', 'Аномалия']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred_label))

In [217]:
class AnomalyPredictor:
    """Ансамблевая модель для предсказания аварий"""
    
    def __init__(self, config: dict):
        self.config = config
        self.lgb_model = None
        self.lstm_model = None
        self.scaler = StandardScaler()
        self.feature_cols = None
        
    def prepare_features(self, df: pd.DataFrame):
        """Подготовка фичей для разных моделей"""
        
        # Общие фичи для LGBM
        feature_engineer = FeatureEngineer()
        data = feature_engineer.create_features(df, self.config['features']['rolling_windows'])
        
        # Выбираем только числовые колонки
        numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
        
        # Убираем целевую и временные метки
        exclude_cols = ['target', 'is_failure', 'time_to_failure', 'ttf_log', 'timestamp']
        self.feature_cols = [col for col in numeric_cols if col not in exclude_cols]
        
        return data[self.feature_cols], data['target'] if 'target' in data.columns else None
    
    def train_lgbm(self, X_train, y_train, X_val, y_val):
        """Обучение LightGBM"""
        
        # Веса классов (аномалий меньше)
        class_weights = len(y_train) / (2 * np.bincount(y_train.astype(int)))
        weight_map = {i: class_weights[i] for i in range(len(class_weights))}
        sample_weights = y_train.map(weight_map)
        
        # Создание датасета
        train_data = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        # Параметры
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': 0,
            'num_threads': 4,
            'min_data_in_leaf': 20
        }
        
        # Обучение
        self.lgb_model = lgb.train(
            params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(50)]
        )
        
        return self.lgb_model
    
    def build_lstm(self, input_shape):
        """Построение LSTM модели"""
        
        model = models.Sequential([
            layers.LSTM(64, return_sequences=True, input_shape=input_shape),
            layers.Dropout(0.2),
            layers.LSTM(32, return_sequences=False),
            layers.Dropout(0.2),
            layers.Dense(16, activation='relu'),
            layers.Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        return model
    
    def train_ensemble(self, df: pd.DataFrame):
        """Обучение ансамбля"""
        
        # Подготовка данных
        X, y = self.prepare_features(df)
        
        # Time Series Split
        tscv = TimeSeriesSplit(n_splits=3)
        
        for train_idx, val_idx in tscv.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            # Масштабирование для LSTM
            X_train_scaled = self.scaler.fit_transform(X_train)
            X_val_scaled = self.scaler.transform(X_val)
            
            # Переформатирование для LSTM [samples, timesteps, features]
            # Используем последовательности по 60 минут
            seq_length = 60
            
            X_train_seq = self.create_sequences(X_train_scaled, seq_length)
            y_train_seq = y_train.iloc[seq_length-1:].values
            
            X_val_seq = self.create_sequences(X_val_scaled, seq_length)
            y_val_seq = y_val.iloc[seq_length-1:].values
            
            # Обучение LSTM
            self.lstm_model = self.build_lstm((seq_length, X_train.shape[1]))
            
            self.lstm_model.fit(
                X_train_seq, y_train_seq,
                validation_data=(X_val_seq, y_val_seq),
                epochs=50,
                batch_size=32,
                verbose=1
            )
            
            # Обучение LGBM
            self.train_lgbm(X_train, y_train, X_val, y_val)
            
            break  # Используем только первый сплит для демо
            
        return self
    
    def create_sequences(self, data, seq_length):
        """Создание последовательностей для LSTM"""
        sequences = []
        for i in range(len(data) - seq_length + 1):
            sequences.append(data[i:i+seq_length])
        return np.array(sequences)
    
    def predict(self, X: pd.DataFrame, use_ensemble: bool = True):
        """Предсказание ансамбля"""
        
        if use_ensemble and self.lgb_model is not None and self.lstm_model is not None:
            # LGBM предсказание
            lgb_proba = self.lgb_model.predict(X)
            
            # LSTM предсказание
            X_scaled = self.scaler.transform(X)
            X_seq = self.create_sequences(X_scaled, 60)
            
            # Для первых 59 точек используем только LGBM
            lstm_proba = np.zeros(len(X))
            if len(X_seq) > 0:
                lstm_pred = self.lstm_model.predict(X_seq).flatten()
                lstm_proba[-len(lstm_pred):] = lstm_pred
                lstm_proba[:60] = lgb_proba[:60]  # Заполняем начало
            
            # Ансамбль: среднее взвешенное
            ensemble_proba = 0.6 * lgb_proba + 0.4 * lstm_proba
            
            return ensemble_proba
        elif self.lgb_model is not None:
            return self.lgb_model.predict(X)
        else:
            raise ValueError("Модель не обучена")